# Error analysis
Test set B has the categories. Will load test set A and B and combine them so that test set B is extended with the sarcasm column.

In [3]:
import pandas as pd
import glob
import os

test_A = pd.read_csv("../data/task_A_En_test.csv")
test_B = pd.read_csv("../data/task_B_En_test.csv")

categories = ["sarcasm", "irony", "satire", "understatement", "overstatement", "rhetorical_question"]

test_B_unique = test_B.drop_duplicates(subset="text")
combined = test_A.merge(test_B_unique, on="text", how="left")

In [5]:
print(len(combined))# should be 1400
print(combined[categories].isna().sum().sum())#should be 0
print(((combined["sarcastic"] == 1) & (combined[categories].sum(axis=1) == 0)).sum())#should be 0

1400
0
0


Number of rows is still 1400, there are no tweets that don't have the category columns, and there are no sarcastic tweets that don't have at least one category match.

In [12]:
files = sorted(glob.glob("../results/*_test_predictions.csv"))

run_columns = []
for f in files:
    run_name = os.path.basename(f).replace("_test_predictions.csv", "")
    predictions = pd.read_csv(f)
    assert (predictions["text"].values == combined["text"].values).all(), f"Wrong order in {f}"
    combined[run_name] = predictions["pred"].values
    run_columns.append(run_name)

print(run_columns)

['baseline', 'bertweet_seed42', 'bertweet_seed43', 'bertweet_seed44', 'modernbert_seed42', 'modernbert_seed43', 'modernbert_seed44', 'qwen7b_0shot_seed42', 'qwen7b_10shot_seed42', 'qwen7b_10shot_seed43', 'qwen7b_10shot_seed44', 'qwen7b_lora_seed42', 'qwen7b_lora_seed43', 'qwen7b_lora_seed44', 'roberta_seed42', 'roberta_seed43', 'roberta_seed44']


In [7]:
rows = []
for run in run_columns:
    model = run.split("_seed")[0]
    for category in categories:
        subset = combined[combined[category] == 1]
        recall = (subset[run] == 1).mean()
        rows.append({"model": model, "run": run, "category": category, "n": len(subset), "recall": recall})

recall_per_run = pd.DataFrame(rows)

In [8]:
recall_table = recall_per_run.groupby(["category", "model"])["recall"].mean().unstack()
recall_table.insert(0, "n", recall_per_run.groupby("category")["n"].first())
print(recall_table.round(3))

model                  n  baseline  bertweet  modernbert  qwen7b_0shot  \
category                                                                 
irony                 20     0.650     0.633       0.617         1.000   
overstatement         10     0.500     0.500       0.733         0.900   
rhetorical_question   11     0.455     0.879       0.818         1.000   
sarcasm              180     0.450     0.687       0.702         0.911   
satire                49     0.286     0.381       0.571         0.776   
understatement         1     0.000     1.000       1.000         1.000   

model                qwen7b_10shot  qwen7b_lora  roberta  
category                                                  
irony                        0.683        0.767    0.567  
overstatement                0.700        0.667    0.467  
rhetorical_question          0.818        0.818    0.727  
sarcasm                      0.744        0.770    0.617  
satire                       0.517        0.585    0.

In [9]:
sarcastic_tweets = combined[combined["sarcastic"] == 1]
missed_by_all = sarcastic_tweets[(sarcastic_tweets[run_columns] == 0).all(axis=1)]

print(f"{len(missed_by_all)} sarcastic tweets were missed by all models")

for _, row in missed_by_all.iterrows():
    cats = [c for c in categories if row[c] == 1]
    print(f"- {row['text']} {cats}")

1 sarcastic tweets were missed by all models
- It's so weird how I look ugly in photos but I also look ugly in the mirror ['sarcasm']


In [20]:
miss_share = (sarcastic_tweets[run_columns] == 0).mean(axis=1)
missed_by_most = sarcastic_tweets[miss_share > 0.5]
print(f"{len(missed_by_most)} sarcastic tweets missed by most models")

for _, row in missed_by_most.iterrows():
    cats = [c for c in categories if row[c] == 1]
    print(f"- {row['text']} {cats}")

48 sarcastic tweets missed by most models
- Sometimes I lay in bed and think about how today will be the day I make my life better. Exercise, drinking water, eating healthy. Then I wake up.  ['sarcasm']
- Max Verstappen is such a clean driver, he never makes dirty moves when racing. ['sarcasm', 'satire']
- Boris looked so statesman-like in that public address ['sarcasm', 'satire']
- I’m soo excited to cook a three course Christmas dinner for my whole family this year!  ['sarcasm', 'satire']
- Thoughts and prayers will actually work this time. Trust me ['sarcasm', 'satire']
- if you listen carefully, you can hear me not carig ['sarcasm']
- Politics is like waves, they depend on the wind where it blows ['sarcasm']
- There are a bazillion things to do in my small town of Wadsworth. One thing you can do is ride horses on the moon and sing songs with Abe Lincoln in the bar. On top of that, I recommend eating a hundred gallons of ice cream at any local ice cream shop downtown. Some of the fl

In [23]:
bertweet_runs = [r for r in run_columns if r.startswith("bertweet")]
missed_by_bertweet = sarcastic_tweets[(sarcastic_tweets[bertweet_runs] == 0).all(axis=1)]
print(f"{len(missed_by_bertweet)} tweets missed by all BERTweet runs (the best performing model)")

for _, row in missed_by_bertweet.iterrows():
    cats = [c for c in categories if row[c] == 1]
    print(f"- {row['text']} {cats}")

47 tweets missed by all BERTweet runs (the best performing model)
- Sometimes I lay in bed and think about how today will be the day I make my life better. Exercise, drinking water, eating healthy. Then I wake up.  ['sarcasm']
- Max Verstappen is such a clean driver, he never makes dirty moves when racing. ['sarcasm', 'satire']
- Boris looked so statesman-like in that public address ['sarcasm', 'satire']
- I’m soo excited to cook a three course Christmas dinner for my whole family this year!  ['sarcasm', 'satire']
- Thoughts and prayers will actually work this time. Trust me ['sarcasm', 'satire']
- if you listen carefully, you can hear me not carig ['sarcasm']
- Politics is like waves, they depend on the wind where it blows ['sarcasm']
- There are a bazillion things to do in my small town of Wadsworth. One thing you can do is ride horses on the moon and sing songs with Abe Lincoln in the bar. On top of that, I recommend eating a hundred gallons of ice cream at any local ice cream shop 